# Single-population Hawkes with QUADRATIC φ, NO RESET: pipeline vs simulation

Builder-generated companion to the linear+reset notebook.  Theory:

  - single population E of size 2
  - quadratic transfer: `φ(v) = a·v²`
  - per-pair exponential filter `g[i,j]` with timescales `taug[i,j]`
  - NO spike reset (voltage drifts continuously through spike events)

**Why this case is interesting.**  With `φ = a·v²` we get a nonzero
`φ''(v*) = 2·a`, which introduces the cubic vertex `nt·δv·δv` at
bigrade (1, 2).  This is the canonical quad-Hawkes case — same
physics as `pipeline.theories.quad_hawkes_2pop_expg`, but loaded
from a builder-generated `.theory.py` file so we can iterate on the
theory definition without touching pipeline code.

**Theory file**: `theories/single_population_quad_exp_test.theory.py`  
**Simulator**:   `models.hawkes_sim_multipop_numba.sim_hawkes_multipop_quad_numba`

**Modes supported** (set in the configuration cell):
- `k = 1, max_ell = 1` — tadpole 1-loop rate shift (bar chart).
- `k = 2, max_ell = 0` — tree-level `C^(2)(τ)` slice.
- `k = 2, max_ell = 1` — tree + 1-loop `C^(2)(τ)` slice.

## 1. Setup

In [ ]:
%display latex
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os, sys, time, importlib, importlib.util
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))

# Pipeline (theory side)
from pipeline import compute_cumulants

# Simulation side — quadratic-rate, no-reset variant.  Per-pair
# exp filter, signed coupling matrix W — same plumbing as the other
# multipop sim variants.
from models.hawkes_sim_multipop_numba import (
    sim_hawkes_multipop_quad_numba,
    build_sim_arrays,
    flat_index_of,
)
from models.cumulant_estimator import compute_kpoint_slice

## 2. Configuration

`fundamental` provides numerical values for every parameter in
`single_population_quad_exp_test.theory.py`.  Names match the
theory file: `Em`, `tau`, `a`, `w`, `taug`.

Pick `k` / `max_ell` for the comparison:
- `k=2, max_ell=0` — tree-level cross-cumulant (LNA prediction).
- `k=2, max_ell=1` — tree + 1-loop from the quad-φ cubic vertex.
- `k=1, max_ell=1` — tadpole shift to the mean rate.

In [ ]:
fundamental = {
    'Em':   [0.8, 0.78],
    'tau':  [10.0, 9.0],
    'a':    [0.44, 0.44],
    'taug': [[2.0, 3.0], [1.0, 3.0]],
    'w':    [[0.25, 0.25], [0.2, 0.3]],
}

k        = 2
max_ell  = 0
# External fields — single-pop natural name is just 'n' (without
# a population suffix); the pipeline maps 'n' → internal 'dn'.
if k == 1:
    external_fields = [('n', 1)]
else:
    external_fields = [('n', 1), ('n', 2)]

tau_max  = 20.0
tau_step = 2.5

PARALLEL  = True
N_WORKERS = None

# ─── Grouped Phase J prototype (opt-in) ───────────────
# When True, groups typed diagrams by (parent prediagram,
# external_legs) and integrates the SUMMED integrand once per group
# instead of once per typed diagram.  Mathematically exact (bitwise
# cross-checked).
USE_GROUPED_PHASE_J = False

# Simulation knobs.  Quadratic rate is more sensitive to large v
# excursions than the linear variant; use a moderate T_sim and
# small dt_sim so the Poisson statistics resolve cleanly.
N_RUNS   = 4
T_sim    = float(2_000_000)
dt_sim   = 0.01            # Euler step
dt_bin   = 0.25            # binning resolution for the cumulant estimator

print(f'k={k}, max_ell={max_ell}, external_fields={external_fields}')
print(f'tau_max={tau_max}, tau_step={tau_step}')
print(f'PARALLEL={PARALLEL}, N_WORKERS={N_WORKERS}, '
      f'USE_GROUPED_PHASE_J={USE_GROUPED_PHASE_J}')
print(f'N_RUNS={N_RUNS}, T_sim={T_sim:.0g}, dt_sim={dt_sim}, dt_bin={dt_bin}')

## 3. Load the theory file

Single population E of size 2, fields `n` and `v`, quadratic
transfer `φ(v) = a·v²`, no spike reset.  Auto-response names are
`nt`, `vt`; auto-saddles are `nstar`, `vstar`.  The MF equations
are `vstar = Em + Σ w·g·nstar` (with `g` integrating to 1) and
`nstar = phi(vstar) = a·vstar²`.

In [ ]:
THEORY_NAME = 'single_population_quad_exp_test'
THEORIES_DIR = os.path.abspath('../theories')

spec = importlib.util.spec_from_file_location(
    f'theories.{THEORY_NAME}',
    os.path.join(THEORIES_DIR, f'{THEORY_NAME}.theory.py'),
)
theory_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(theory_mod)

model = theory_mod.build()
print(f'Loaded theory: {model["name"]!r}')
print(f'Populations: {[(p["name"], p["size"]) for p in model["populations"]]}')
print(f'Fields (physical): '
      f'{[f["name"] for f in model["physical_fields"]]}')
print(f'Kernels: {[k["name"] for k in model["kernels"]]}')
print(f'Functions: '
      f'{[(fn["name"], fn.get("args")) for fn in model["functions"]]}')

## 3.5 Diagnostic: source + vertex types

Prints every source vertex and interaction vertex with bigrade,
leg structure, raw + `simplify_full()`-cleaned coefficient.  For
quadratic φ without reset you should see:

- `(k, 0)` sources for k=2..taylor with coeff `∝ a·vstar²`
- `(1, 2)` cubic vertex with coeff `-a` — the curvature term that
  drives 1-loop corrections in this theory.
- NO `(1, 2)` vertex with coeff `∝ tau` (that would be a spike-reset
  signature, which this theory does NOT have).

In [ ]:
from msrjd.core.field_theory import FieldTheory
from msrjd.core.vertices import (extract_source_types,
                                  extract_vertex_types,
                                  NoiseSourceType)
from sage.all import SR

_taylor_order = max(k + 2 * max_ell, 4)
_ft_diag = FieldTheory(model, taylor_order=_taylor_order)
_ft_diag.expand()
print(f'FieldTheory.expand done (taylor_order={_taylor_order})')

_sources = extract_source_types(_ft_diag)
_vertices = extract_vertex_types(_ft_diag)
print(f'\n  {len(_sources)} source vertices, {len(_vertices)} interaction vertices')

from collections import Counter
src_bigrades = Counter(s.bigrade for s in _sources)
vt_bigrades  = Counter(v.bigrade for v in _vertices)
print(f'\n  Source bigrades:      {dict(sorted(src_bigrades.items()))}')
print(f'  Interaction bigrades: {dict(sorted(vt_bigrades.items()))}')

def _coeff_diag(label, idx, obj):
    raw = SR(obj.coefficient)
    raw_str = str(raw)
    try:
        simp = raw.simplify_full()
    except Exception:
        simp = raw
    simp_str = str(simp)
    legs_str = (f'resp={obj.response_legs}'
                + (f', phys={obj.physical_legs}'
                   if hasattr(obj, 'physical_legs') else ''))
    print(f'  {label}[{idx}]  bigrade={obj.bigrade}  {legs_str}')
    print(f'    raw  ({len(raw_str):4d} chars): {raw_str}')
    if simp_str != raw_str:
        delta = len(raw_str) - len(simp_str)
        sign  = '−' if delta > 0 else '+'
        print(f'    simp ({len(simp_str):4d} chars, {sign}{abs(delta):3d}): {simp_str}')
    residual = []
    for v in raw.variables():
        sname = str(v)
        if sname.startswith(('phi', 'z_g_', '_v_mf_', '_phi_arg_')):
            residual.append(sname)
    if residual:
        print(f'    ⚠ residual formal symbols: {sorted(set(residual))}')

print('\n=== Source vertices ===')
for idx, src in enumerate(_sources):
    _coeff_diag('src', idx, src)

print('\n=== Interaction vertex types ===')
for idx, vt in enumerate(_vertices):
    _coeff_diag('vt ', idx, vt)

## 4. Theory side — one pipeline call

Runs the full MSR-JD chain.  At `max_ell=0` the tree-level
$C^{(2)}(\tau)$ matches the LNA prediction (correct gain
$\phi'(v^*) = 2 a v^*$); at `max_ell=1` the 1-loop contribution
from the cubic vertex (coeff $-a$) is added.

In [ ]:
t0 = time.perf_counter()
th = compute_cumulants(
    model               = model,
    k                   = k,
    max_ell             = max_ell,
    fundamental         = fundamental,
    external_fields     = external_fields,
    tau_max             = tau_max,
    tau_step            = tau_step,
    parallel            = PARALLEL,
    n_workers           = N_WORKERS,
    use_grouped_phase_j = USE_GROUPED_PHASE_J,
    verbose             = True,
)
print(f'\nTheory side took {time.perf_counter() - t0:.1f}s')

tau_grid_th    = th['tau_grid']
C_theory_total = th['C_tau'].real
C_by_ell       = th['C_tau_by_ell']
C_theory_tree  = (C_by_ell[0].real if 0 in C_by_ell
                  else np.zeros_like(C_theory_total))
C_theory_loop  = C_theory_total - C_theory_tree

mf_values = th['mf_values']
print('\nMean-field saddles:')
for name, vals in mf_values.items():
    print(f'  {name!r:8} = {vals}')

# Quick self-consistency check: nstar = a · vstar².
nstar = np.array(mf_values['nstar'])
vstar = np.array(mf_values['vstar'])
a_arr = np.array(fundamental['a'])
phi_v = a_arr * vstar**2
print(f'\nphi(vstar) = a·vstar² = {phi_v.tolist()}')
print(f'closure residual (nstar - a·vstar²) = '
      f'{(nstar - phi_v).tolist()}')

print(f'\nTotal diagrams: {len(th["diagrams"])}')
n_per_ell = {ell: sum(1 for r in th['diagrams'] if r['ell'] == ell)
             for ell in sorted({r['ell'] for r in th['diagrams']})}
for ell, n_d in n_per_ell.items():
    print(f'    ell={ell}: {n_d} diagrams')

## 5. Simulation side

Quadratic-rate, no-reset simulator with per-pair filters.  All 2
neurons are stacked into a flat array.  Per Euler step:

$$\lambda_i = \max(a_i \cdot v_i^2, 0)$$
$$n_i \sim \mathrm{Poisson}(\lambda_i\,\mathrm{d}t)$$
$$\tau_{g,ij}\,\dot F_{ij} + F_{ij} = n_j$$
$$\tau_{v,i}\,\dot v_i = -v_i + E_i + \sum_j W_{ij}\,F_{ij}$$

Voltage drifts continuously through spike events (no reset).

In [ ]:
import secrets as _secrets

arr = build_sim_arrays(model, fundamental, mf_values)
N           = arr['N']
tau_v       = arr['tau_v']
a_gain      = arr['a_gain']
E_drive     = arr['E_drive']
W           = arr['W']
tau_g_arr   = arr['tau_g']
v_init      = arr['v_init']
pop_offsets = arr['pop_offsets']

print(f'Stacked neuron count: N = {N}')
for pname, (start, size) in pop_offsets.items():
    print(f'  pop {pname!r}: flat indices [{start}, {start + size})')
print(f'tau_v   = {tau_v}')
print(f'a_gain  = {a_gain}')
print(f'E_drive = {E_drive}')
print(f'v_init  = {v_init}')
print(f'a · v_init² (≈ MF rate, quad φ) = {a_gain * v_init**2}')
print(f'W:\n{W}')
print(f'tau_g:\n{tau_g_arr}')

In [ ]:
# Discretization
n_steps        = int(T_sim / dt_sim)
bin_size_steps = max(int(round(dt_bin / dt_sim)), 1)
dt_bin_eff     = bin_size_steps * dt_sim
n_bins         = n_steps // bin_size_steps
max_lag_bins   = int(tau_max / dt_bin_eff)
tau_sim_grid   = np.arange(-max_lag_bins, max_lag_bins + 1) * dt_bin_eff

pop_indices = [flat_index_of(model, pop_offsets, ef[0], ef[1])
               for ef in external_fields]
field_types = [ef[0] for ef in external_fields]
print(f'External fields {external_fields} → flat sim indices {pop_indices}')

# JIT warmup
_ = sim_hawkes_multipop_quad_numba(
    int(1000), float(dt_sim),
    tau_v, a_gain, E_drive,
    W, tau_g_arr,
    v_init.copy(),
    int(bin_size_steps), int(100), int(0),
)
print('JIT warmup done.')

In [ ]:
BASE_SEED = _secrets.randbits(31)
C_sim_runs   = []
rate_runs    = []
voltage_runs = []

t0 = time.perf_counter()
for run in range(N_RUNS):
    seed = int(BASE_SEED + run)
    binned_counts, voltage_bins, total_spikes = sim_hawkes_multipop_quad_numba(
        int(n_steps), float(dt_sim),
        tau_v, a_gain, E_drive,
        W, tau_g_arr,
        v_init.copy(),
        int(bin_size_steps), int(n_bins), seed,
    )
    rate_runs.append([float(total_spikes[i]) / T_sim for i in range(N)])
    voltage_runs.append(voltage_bins.mean(axis=1))

    if k == 1:
        pass
    else:
        tau_run, C_run = compute_kpoint_slice(
            binned_counts, float(dt_bin_eff),
            [int(p) for p in pop_indices],
            [0, None], int(max_lag_bins),
            field_types=field_types,
            voltage_bins=voltage_bins,
        )
        C_sim_runs.append(C_run)
    print(f'  run {run+1}/{N_RUNS}: rates = '
          f'{[f"{r:.4f}" for r in rate_runs[-1]]}')

if k >= 2:
    C_sim_runs = np.array(C_sim_runs)
    C_sim_mean = C_sim_runs.mean(axis=0)
    C_sim_sem  = C_sim_runs.std(axis=0, ddof=1) / np.sqrt(N_RUNS)
else:
    C_sim_runs = C_sim_mean = C_sim_sem = None

rate_runs_arr = np.array(rate_runs)
rate_sim_mean = rate_runs_arr.mean(axis=0)
rate_sim_sem  = rate_runs_arr.std(axis=0, ddof=1) / np.sqrt(N_RUNS)
vmean_sim     = np.array(voltage_runs).mean(axis=0)

print(f'\nSimulation side took {time.perf_counter() - t0:.1f}s '
      f'({N_RUNS} runs × T={T_sim:.0g})')
print(f'  Sim mean rates (per neuron): {rate_sim_mean}')

# Build the theory-MF nstar vector in the SAME flat order as the sim.
nstar_flat = np.zeros(N)
for pname, (start, size) in pop_offsets.items():
    for key in (f'n{pname}star', 'nstar'):
        if key in mf_values:
            for i in range(size):
                nstar_flat[start + i] = mf_values[key][i]
            break
print(f'  Theory n* (flat order):      {nstar_flat}')

if k == 1:
    mid = len(tau_grid_th) // 2
    rate_theory_tree     = float(nstar_flat[pop_indices[0]])
    rate_theory_correction = sum(
        float(C_by_ell[ell].real[mid]) for ell in C_by_ell if ell >= 1
    )
    rate_theory_total = rate_theory_tree + rate_theory_correction
    print()
    print(f'k=1 picked external field {external_fields[0]} '
          f'→ flat index {pop_indices[0]}')
    print(f'  sim_rate                    = {rate_sim_mean[pop_indices[0]]:.6f}'
          f'   (SEM {rate_sim_sem[pop_indices[0]]:.2e})')
    print(f'  theory tree (= n*)          = {rate_theory_tree:.6f}')
    print(f'  theory 1-loop correction    = {rate_theory_correction:+.6f}')
    print(f'  theory tree + loops total   = {rate_theory_total:.6f}')

## 6. Theory vs simulation

**For k = 2**: per-neuron rate bars (sim vs MF) + the C^(2)(τ)
slice with tree-only and tree+loop curves.

**For k = 1**: bar chart showing per-neuron sim, tree (= n*),
tree + loop rate for the configured external field.

In [ ]:
if k == 1:
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    x = np.arange(N)
    width = 0.27
    tick_labels = []
    for pname, (start, size) in pop_offsets.items():
        for i in range(size):
            tick_labels.append(f'{pname}{i+1}')

    rate_tree_plus_loop = nstar_flat.copy()
    rate_tree_plus_loop[pop_indices[0]] += rate_theory_correction

    ax.bar(x - width, rate_sim_mean, width,
           yerr=rate_sim_sem, capsize=3,
           label='Simulation', color='#2ECC71', edgecolor='black')
    ax.bar(x,         nstar_flat, width,
           label='Theory: tree (= $n^*$)',
           color='#3F00FF', edgecolor='black', alpha=0.85)
    ax.bar(x + width, rate_tree_plus_loop, width,
           label=f'Theory: tree + {max_ell}-loop',
           color='#E74C3C', edgecolor='black', alpha=0.85)
    ax.annotate(
        f'1-loop computed for {external_fields[0][0]}{external_fields[0][1]}',
        xy=(pop_indices[0] + width, rate_tree_plus_loop[pop_indices[0]]),
        xytext=(pop_indices[0] + width + 0.2,
                rate_tree_plus_loop[pop_indices[0]] * 1.15),
        fontsize=9,
        arrowprops=dict(arrowstyle='->', color='#555'),
    )
    ax.set_xticks(x)
    ax.set_xticklabels(tick_labels)
    ax.set_ylabel('Firing rate')
    field_a = external_fields[0]
    ax.set_title(f'k=1 rate (quad φ, no reset), max_ell={max_ell}, '
                 f'loop target: {field_a[0]}_{field_a[1]}')
    ax.legend()
    ax.grid(True, axis='y', alpha=0.25)
    fig.tight_layout()
    plt.show()
else:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5),
                             gridspec_kw={'width_ratios': [1, 2]})

    ax_bar = axes[0]
    x = np.arange(N)
    width = 0.4
    ax_bar.bar(x - width/2, rate_sim_mean, width,
               yerr=rate_sim_sem, capsize=3,
               label='Simulation', color='#2ECC71', edgecolor='black')
    ax_bar.bar(x + width/2, nstar_flat, width,
               label='Mean-field', color='#3498DB', edgecolor='black')
    ax_bar.set_xticks(x)
    tick_labels = []
    for pname, (start, size) in pop_offsets.items():
        for i in range(size):
            tick_labels.append(f'{pname}{i+1}')
    ax_bar.set_xticklabels(tick_labels)
    ax_bar.set_ylabel('Firing rate')
    ax_bar.set_title('Mean firing rates per neuron')
    ax_bar.legend()
    ax_bar.grid(True, axis='y', alpha=0.25)

    ax = axes[1]
    ax.plot(tau_sim_grid, C_sim_mean, color='#1f1f1f', linewidth=1.4,
            label=f'Simulation ({N_RUNS} runs avg)', alpha=0.85)
    ax.fill_between(tau_sim_grid,
                    C_sim_mean - C_sim_sem,
                    C_sim_mean + C_sim_sem,
                    color='#1f1f1f', alpha=0.15, label='Sim SEM')
    ax.plot(tau_grid_th, C_theory_tree, color='#3F00FF', linewidth=1.6,
            linestyle='--', label='Theory: tree only')
    if max_ell > 0:
        ax.plot(tau_grid_th, C_theory_total, color='#E74C3C',
                linewidth=1.6, label=f'Theory: tree + {max_ell}-loop')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    field_a, field_b = external_fields
    ax.set_xlabel(r'$\tau$')
    ax.set_ylabel(r'$C^{(2)}$')
    ax.set_title(f'Cross-cumulant (quad φ, no reset): '
                 f'$\\langle\\delta {field_a[0]}_{{{field_a[1]}}}(0)\\,'
                 f'\\delta {field_b[0]}_{{{field_b[1]}}}(\\tau)\\rangle_c$')
    ax.legend()
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    plt.show()

## 7. Numerical residual

**k = 2**: tree-only vs tree+loop residual.

**k = 1**: rate-shift residual.

In [ ]:
if k == 1:
    i_tgt   = pop_indices[0]
    r_sim   = float(rate_sim_mean[i_tgt])
    r_sem   = float(rate_sim_sem[i_tgt])
    r_tree  = rate_theory_tree
    r_total = rate_theory_total

    tree_resid     = r_sim - r_tree
    treeloop_resid = r_sim - r_total

    print(f'Target field          : {external_fields[0]} '
          f'(flat sim index {i_tgt})')
    print(f'Sim rate              : {r_sim:.6f}  (SEM {r_sem:.2e})')
    print(f'Theory tree (= n*)    : {r_tree:.6f}')
    print(f'Theory tree + loops   : {r_total:.6f}')
    print()
    print(f'Sim − tree            : {tree_resid:+.4e}  '
          f'({tree_resid / r_sem:+.2f}σ)')
    if max_ell >= 1:
        print(f'Sim − (tree+loop)     : {treeloop_resid:+.4e}  '
              f'({treeloop_resid / r_sem:+.2f}σ)')
        print(f'|residual| reduction  : '
              f'{abs(tree_resid) - abs(treeloop_resid):+.4e}  '
              f'(positive ⇒ loop correction shrank the gap)')
else:
    C_total_on_sim_grid = np.interp(tau_sim_grid, tau_grid_th,
                                    C_theory_total)
    residual            = C_sim_mean - C_total_on_sim_grid

    peak        = max(abs(C_sim_mean.max()), abs(C_sim_mean.min()))
    rms_rel     = float(np.sqrt(np.mean(residual**2)) / peak)
    max_abs_rel = float(np.max(np.abs(residual)) / peak)
    sem_peak    = float(C_sim_sem[np.argmax(np.abs(C_sim_mean))])

    print(f'Sim peak |C|             : {peak:+.4e}')
    print(f'Residual RMS / peak      : {rms_rel:.3%}')
    print(f'Residual max / peak      : {max_abs_rel:.3%}')
    print(f'Sim SEM at peak          : {sem_peak:+.3e} '
          f'({sem_peak / peak:.3%} of peak)')

    if max_ell > 0:
        C_tree_on_sim_grid = np.interp(tau_sim_grid, tau_grid_th,
                                       C_theory_tree)
        tree_residual      = C_sim_mean - C_tree_on_sim_grid
        tree_rms_rel       = float(np.sqrt(np.mean(tree_residual**2))
                                   / peak)
        print()
        print(f'Tree-only residual RMS / peak  : {tree_rms_rel:.3%}')
        print(f'Tree+loop residual RMS / peak  : {rms_rel:.3%}'
              f'   ← Δ = {tree_rms_rel - rms_rel:+.3%}')

## 8. (Optional) Save outputs

In [ ]:
from pipeline import save_npz, save_csv

SAVE = False

if SAVE:
    out_dir = '../pipeline_outputs/singlepop_quad_sim_compare'
    os.makedirs(out_dir, exist_ok=True)
    leg_tag = '_'.join(f'{ef[0]}{ef[1]}' for ef in external_fields)
    slug = f'singlepop_quad_{leg_tag}_k{k}_ell{max_ell}'

    sim_extra = {
        'rates_sim_mean'  : rate_sim_mean,
        'rates_sim_sem'   : rate_sim_sem,
        'sim_N_RUNS'      : np.array([N_RUNS], dtype=int),
        'sim_T'           : np.array([T_sim]),
        'sim_dt'          : np.array([dt_sim]),
        'sim_dt_bin'      : np.array([dt_bin]),
        'sim_variant'     : np.array(['quad_no_reset']),
        'pop_offsets_keys': np.array(list(pop_offsets.keys())),
        'pop_offsets_vals': np.array([list(v) for v in pop_offsets.values()]),
    }
    if k >= 2:
        sim_extra.update({
            'tau_grid_sim' : tau_sim_grid,
            'C_sim_mean'   : C_sim_mean,
            'C_sim_sem'    : C_sim_sem,
        })
    npz_path = f'{out_dir}/{slug}.npz'
    csv_path = f'{out_dir}/{slug}.csv'
    save_npz(th, npz_path, extra=sim_extra)
    save_csv(th, csv_path)
    print(f'Saved: {npz_path}')
    print(f'Saved: {csv_path}')
else:
    print('SAVE=False — outputs not written.  Flip the flag above to save.')

---

### Notes specific to quadratic φ without reset

- **Nonzero φ curvature.**  With $\phi(v) = a\,v^2$, $\phi''(v^*) = 2a$,
  so the cubic vertex $-a\,\tilde n\,\delta v\,\delta v$ at bigrade
  $(1,2)$ is present.  This is the canonical source of 1-loop
  corrections in quad-Hawkes — turn on `max_ell = 1` to see them.
- **MF closure** $n^* = a \cdot v^{*2}$ should hold to floating-point
  precision at the saddle (printed in section 4).
- **No reset.**  Voltage drifts continuously through spike events
  on the sim side, and the action contains no $+\tau v n$ term
  on the theory side.  If you see a $(1, 2)$ vertex with coeff
  $\propto \tau$ in the §3.5 diagnostic, the theory file has a
  reset term it shouldn't.
- **Single-pop parameter naming.**  Theory uses bare `tau`, `a`,
  `Em`, `w`, `taug` (no population suffix).  `build_sim_arrays`
  resolves them via the `name_or_suffixed` fallback.